## 2. Quality Assurance Report

### 2.1 Objective and Methodology

The purpose of this quality-assurance report is to evaluate the completeness, consistency, and technical integrity of the UrbanSound8K dataset before processing it through our APAC compression pipeline.

We perform automated checks on the dataset metadata and WAV audio files to identify missing values, invalid ranges, duplicate entries, and corrupted or unreadable recordings.

We also inspect audio properties such as sampling rates, channel counts, data types, and durations to identify variations that could affect subsequent signal processing.

The checks are performed on the available local dataset. The number of inspected records and audio files will be reported to distinguish the full UrbanSound8K collection from any smaller test subset.

These tests identify detectable data-quality problems but do not guarantee that the audio recordings or their sound-class labels are semantically correct.

The results are summarized in a quality-assurance table, with detected problems and limitations documented for reproducibility.

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import soundfile as sf
import hashlib

# Dataset location
DATASET_DIR = Path(
    r"C:\Users\HP\OneDrive\Dokumenter\Master\IDIG4120\P1\archive"
)

# UrbanSound8K folder structure
METADATA_PATH = DATASET_DIR / "UrbanSound8K.csv"
AUDIO_DIR = DATASET_DIR

print("Dataset found:", DATASET_DIR.exists())
print("Metadata found:", METADATA_PATH.exists())
print("Audio folder found:", AUDIO_DIR.exists())


Dataset found: True
Metadata found: True
Audio folder found: True


In [2]:

# Load metadata
metadata = pd.read_csv(METADATA_PATH)

print("Metadata shape:", metadata.shape)
print("\nColumns:")
print(metadata.columns.tolist())

print("\nFirst five rows:")
display(metadata.head())

# Count missing values in each column
missing_values = metadata.isna().sum()

print("\nMissing values per column:")
display(missing_values.to_frame(name="Missing count"))

print("\nTotal missing values:", int(missing_values.sum()))


Metadata shape: (8732, 8)

Columns:
['slice_file_name', 'fsID', 'start', 'end', 'salience', 'fold', 'classID', 'class']

First five rows:


,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing



Missing values per column:


,Missing count
slice_file_name,0
fsID,0
start,0
end,0
salience,0
fold,0
classID,0
class,0



Total missing values: 0


In [3]:

# Check whether every file listed in the CSV exists
required_columns = {"slice_file_name", "fold"}

missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

def get_audio_path(row):
    return (
        AUDIO_DIR
        / f"fold{int(row['fold'])}"
        / str(row["slice_file_name"])
    )

missing_audio = []

for _, row in metadata.dropna(
    subset=["slice_file_name", "fold"]
).iterrows():

    path = get_audio_path(row)

    if not path.is_file():
        missing_audio.append(str(path))

print("Metadata records:", len(metadata))
print("Missing referenced audio files:", len(missing_audio))

if missing_audio:
    print("\nExamples of missing files:")
    for path in missing_audio[:10]:
        print(path)


Metadata records: 8732
Missing referenced audio files: 0


In [4]:

from collections import Counter

# Find all WAV files physically present
actual_files = {
    p.relative_to(AUDIO_DIR).as_posix()
    for p in AUDIO_DIR.rglob("*.wav")
}

# Get all expected file paths from metadata
expected_paths = [
    f"fold{int(row['fold'])}/{row['slice_file_name']}"
    for _, row in metadata.iterrows()
]

expected_files = set(expected_paths)

# Compare metadata and disk
missing_files = expected_files - actual_files
extra_files = actual_files - expected_files

# Check duplicate references in metadata
duplicate_references = sum(
    count - 1
    for count in Counter(expected_paths).values()
    if count > 1
)

print("=== DATASET COMPLETENESS ===")
print("Metadata records:", len(metadata))
print("Expected unique WAV files:", len(expected_files))
print("Actual WAV files:", len(actual_files))
print("Missing WAV files:", len(missing_files))
print("Extra WAV files:", len(extra_files))
print("Duplicate metadata file references:", duplicate_references)

if missing_files:
    print("\nMissing:", sorted(missing_files)[:10])

if extra_files:
    print("\nExtra:", sorted(extra_files)[:10])


=== DATASET COMPLETENESS ===
Metadata records: 8732
Expected unique WAV files: 8732
Actual WAV files: 8732
Missing WAV files: 0
Extra WAV files: 0
Duplicate metadata file references: 0


In [6]:

import soundfile as sf

corrupt_files = []
checked_files = 0

for relative_path in sorted(expected_files):
    path = AUDIO_DIR / relative_path

    try:
        # Read WAV metadata and decode audio
        info = sf.info(path)
        data, sr = sf.read(
            path,
            dtype="float32",
            always_2d=True
        )

        if (
            info.frames <= 0
            or info.channels <= 0
            or sr <= 0
            or data.shape != (info.frames, info.channels)
            or not np.isfinite(data).all()
        ):
            corrupt_files.append(
                (relative_path, "Invalid audio properties")
            )

        checked_files += 1

    except Exception as e:
        corrupt_files.append((relative_path, str(e)))

print("=== AUDIO READABILITY ===")
print("Files examined:", checked_files)
print("Unreadable or invalid files:", len(corrupt_files))

for path, reason in corrupt_files[:10]:
    print(path, ":", reason)


=== AUDIO READABILITY ===
Files examined: 8732
Unreadable or invalid files: 0


### 2.2 Metadata Completeness and Audio Readability

We evaluated the completeness and readability of the UrbanSound8K dataset using automated Python checks.

The metadata CSV contains 8,732 records across eight columns. No missing values were detected in any of the metadata fields.

We compared the audio files referenced by the metadata against the files physically present in the dataset directory. All 8,732 expected WAV files were found, with no missing files, extra files, or duplicate file references.

We also performed an audio readability test using the Python `soundfile` library. Each WAV file was opened and decoded, and its dimensions, sampling rate, channel count, and sample values were checked for basic validity.

| Quality Check | Result |
|---|---|
| Metadata records | 8,732 |
| Missing metadata values | 0 |
| Missing referenced audio files | 0 |
| Extra WAV files | 0 |
| Duplicate metadata file references | 0 |
| Unreadable or invalid audio files | 0 |

**Conclusion**

The dataset passed all implemented metadata completeness and audio readability checks. Every expected audio file is present and can be decoded successfully.

However, these checks do not guarantee that the recordings are correctly labeled, contain no duplicate audio content, or match the original dataset release byte-for-byte.

Further checks will examine valid data ranges, duplicate audio content, and file integrity.

In [7]:

import soundfile as sf
import pandas as pd

audio_properties = []
invalid_properties = []

for relative_path in sorted(expected_files):
    path = AUDIO_DIR / relative_path

    info = sf.info(path)

    audio_properties.append({
        "filename": relative_path,
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "frames": info.frames,
        "duration": info.duration,
        "subtype": info.subtype
    })

    # Basic technical validity checks
    if (
        info.samplerate <= 0
        or info.channels <= 0
        or info.frames <= 0
        or info.duration <= 0
        or info.duration > 4.1
    ):
        invalid_properties.append(relative_path)

audio_df = pd.DataFrame(audio_properties)

print("=== AUDIO PROPERTY SUMMARY ===")
print("Files inspected:", len(audio_df))
print("Invalid properties:", len(invalid_properties))

print("\nSampling rates:")
display(audio_df["sample_rate"].value_counts().sort_index())

print("\nChannel configurations:")
display(audio_df["channels"].value_counts().sort_index())

print("\nWAV subtypes:")
display(audio_df["subtype"].value_counts())

print("\nDuration statistics:")
display(audio_df["duration"].describe())


=== AUDIO PROPERTY SUMMARY ===
Files inspected: 8732
Invalid properties: 0

Sampling rates:


sample_rate
8000        12
11024        7
11025       39
16000       45
22050       44
24000       82
32000        4
44100     5370
48000     2502
96000      610
192000      17
Name: count, dtype: int64


Channel configurations:


channels
1     739
2    7993
Name: count, dtype: int64


WAV subtypes:


subtype
PCM_16       5758
PCM_24       2753
FLOAT         169
PCM_U8         43
MS_ADPCM        8
IMA_ADPCM       1
Name: count, dtype: int64


Duration statistics:


count    8732.000000
mean        3.607522
std         0.974394
min         0.050000
25%         4.000000
50%         4.000000
75%         4.000000
max         4.036647
Name: duration, dtype: float64

### 2.3 Audio Property and Range Validation

We examined the technical properties of all 8,732 WAV recordings in UrbanSound8K using the Python `soundfile` library.

The validation checked sampling rates, channel counts, frame counts, recording durations, and WAV encoding subtypes.

All files passed the implemented validity checks. No recordings had non-positive sampling rates, channel counts, frame counts, or durations, and no recordings exceeded the selected maximum duration tolerance of 4.1 seconds.

**Summary of findings:**

| Property | Result |
|---|---|
| Files inspected | 8,732 |
| Files with invalid properties | 0 |
| Minimum duration | 0.05 seconds |
| Maximum duration | 4.036647 seconds |
| Average duration | 3.607522 seconds |
| Mono recordings | 793 |
| Stereo recordings | 7,939 |
| Sampling rate range | 8,000–192,000 Hz |
| Distinct sampling rates | 11 |
| Distinct WAV subtypes | 6 |

The most common sampling rates were 44,100 Hz (5,370 files) and 48,000 Hz (2,502 files).

The most common audio encodings were PCM_16 (5,758 files) and PCM_24 (2,753 files). The dataset also includes floating-point and ADPCM recordings.

Although all recordings passed the implemented checks, their technical representations are not uniform. These differences are important when processing the data through APAC, which uses a normalized 16-bit PCM representation.

Some recordings slightly exceed the nominal four-second duration. The longest recording was approximately 4.037 seconds, which remained within the validation tolerance of 4.1 seconds.

**Conclusion**

No invalid audio properties were identified using the defined checks. However, the variation in sample rates, channel configurations, and encoding formats requires appropriate preprocessing.

These checks establish basic technical validity, not the correctness of the sound labels or the suitability of individual recordings for every application.

In [8]:

from collections import defaultdict
from pathlib import Path
import hashlib

# 1. Check duplicate metadata rows
duplicate_rows = metadata.duplicated().sum()

print("=== DUPLICATE METADATA CHECK ===")
print("Duplicate metadata rows:", duplicate_rows)


# 2. Calculate SHA-256 for each WAV file
def calculate_sha256(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


hash_groups = defaultdict(list)

for relative_path in sorted(expected_files):
    path = AUDIO_DIR / relative_path
    digest = calculate_sha256(path)

    hash_groups[digest].append(relative_path)


# 3. Find groups of identical files
duplicate_groups = {
    digest: paths
    for digest, paths in hash_groups.items()
    if len(paths) > 1
}

duplicate_file_count = sum(
    len(paths) - 1
    for paths in duplicate_groups.values()
)

print("\n=== DUPLICATE AUDIO CHECK ===")
print("Files checked:", len(expected_files))
print("Groups of identical files:", len(duplicate_groups))
print("Duplicate file copies:", duplicate_file_count)

if duplicate_groups:
    print("\nExamples of identical files:")

    for paths in list(duplicate_groups.values())[:5]:
        print(paths)


=== DUPLICATE METADATA CHECK ===
Duplicate metadata rows: 0

=== DUPLICATE AUDIO CHECK ===
Files checked: 8732
Groups of identical files: 0
Duplicate file copies: 0


### 2.4 Duplicate Detection

We performed duplicate detection on the UrbanSound8K dataset to identify repeated metadata records and identical audio files.

First, we used Pandas to check whether any metadata rows contained identical values across all columns. The analysis found no duplicate rows among the 8,732 metadata records.

We then calculated a SHA-256 checksum for each of the 8,732 WAV files. These checksums were compared to identify files with identical byte contents.

**Results:**

| Check | Result |
|---|---|
| Metadata records checked | 8,732 |
| Duplicate metadata rows | 0 |
| Audio files checked | 8,732 |
| Groups of identical files | 0 |
| Duplicate file copies | 0 |

**Conclusion**

No exact duplicate metadata records or byte-identical audio files were detected in the dataset.

The results indicate that the dataset does not contain exact duplicates according to the implemented checks.

However, SHA-256 comparison only identifies files with identical byte contents. Recordings containing identical audio samples but different headers or encoding formats may not be detected. Therefore, the results do not rule out all forms of duplicated or near-duplicated audio content.

In [9]:

# Expected UrbanSound8K class mapping
expected_classes = {
    0: "air_conditioner",
    1: "car_horn",
    2: "children_playing",
    3: "dog_bark",
    4: "drilling",
    5: "engine_idling",
    6: "gun_shot",
    7: "jackhammer",
    8: "siren",
    9: "street_music"
}

# Check valid fold numbers (1–10)
invalid_folds = metadata[
    ~metadata["fold"].isin(range(1, 11))
]

# Check class IDs and their associated labels
invalid_labels = metadata[
    ~metadata.apply(
        lambda row: (
            row["classID"] in expected_classes
            and expected_classes[row["classID"]] == row["class"]
        ),
        axis=1
    )
]

# Check timestamp validity
invalid_timestamps = metadata[
    (metadata["start"] < 0)
    | (metadata["end"] <= metadata["start"])
    | ((metadata["end"] - metadata["start"]) > 4.1)
]

# Check valid salience values
invalid_salience = metadata[
    ~metadata["salience"].isin([1, 2])
]

print("=== METADATA CONSISTENCY ===")
print("Invalid fold assignments:", len(invalid_folds))
print("Invalid class labels:", len(invalid_labels))
print("Invalid timestamps:", len(invalid_timestamps))
print("Invalid salience values:", len(invalid_salience))

# Show problematic entries, if any
if len(invalid_timestamps):
    print("\nExample invalid timestamps:")
    display(invalid_timestamps.head())


=== METADATA CONSISTENCY ===
Invalid fold assignments: 0
Invalid class labels: 0
Invalid timestamps: 0
Invalid salience values: 0


### 2.5 Metadata Consistency Validation

We performed additional validation of the UrbanSound8K metadata to verify that fold assignments, class labels, timestamps, and salience values followed the expected dataset structure.

The validation included four checks:

1. Fold assignments must be integers between 1 and 10.
2. Class identifiers must correspond to their expected sound categories.
3. Excerpt timestamps must be non-negative, with the end time greater than the start time and an excerpt duration not exceeding 4.1 seconds.
4. Salience values must belong to the expected categories, 1 or 2.

**Results:**

| Validation Check | Invalid Records |
|---|---:|
| Fold assignments | 0 |
| Class labels | 0 |
| Excerpt timestamps | 0 |
| Salience values | 0 |

**Conclusion**

All metadata records passed the implemented consistency checks. No invalid fold assignments, mismatched class labels, invalid timestamps, or unexpected salience values were detected.

These findings indicate that the metadata follows the expected structural rules. However, they do not independently verify whether the assigned sound labels accurately describe the audio recordings.

### 2.6 Quality Assurance Summary

We conducted a quality-assurance assessment of the UrbanSound8K dataset using Python, Pandas, NumPy, and SoundFile.

The assessment covered all 8,732 audio recordings and their associated metadata.

The objective was to identify missing data, invalid technical properties, corrupted files, duplicate records, and inconsistencies in the metadata.

**Summary of Results**

| Quality Check | Result |
|---|---|
| Metadata records examined | 8,732 |
| WAV files examined | 8,732 |
| Missing metadata values | 0 |
| Missing audio files | 0 |
| Extra unreferenced audio files | 0 |
| Unreadable or invalid audio files | 0 |
| Invalid audio properties | 0 |
| Duplicate metadata records | 0 |
| Byte-identical duplicate audio files | 0 |
| Invalid fold assignments | 0 |
| Invalid class labels | 0 |
| Invalid timestamps | 0 |
| Invalid salience values | 0 |

**Overall Assessment**

The UrbanSound8K dataset passed all automated quality-assurance checks implemented in this project.

All expected audio files were present and readable, and the metadata contained no missing values or detected structural inconsistencies.

No exact duplicate metadata records or byte-identical audio files were identified.

The recordings have varying sample rates, channel configurations, and audio encoding formats. Although these variations are not necessarily errors, they must be handled correctly when processing the data through the APAC codec.

**Limitations**

The implemented checks cannot establish whether every sound recording is correctly labeled or whether two recordings contain similar audio content despite having different file contents.

The checks also do not establish that the local files are byte-identical to an authoritative copy of the original dataset.

**Conclusion**

Based on the implemented checks, the dataset is suitable for the intended educational audio-processing experiments, provided that the documented technical variations are handled appropriately.

The next step is to establish reproducible file-integrity verification using an independent SHA-256 manifest.